v61 
- **Verified all metric calculation with Excel** 
- Updated core.analyzer
- Replaced the `Result` pattern with exceptions and flattened the logic
- Refactored the `AlphaEngine` into a streamlined Orchestrator pattern
-  **Strict Date Logic:** No more "Time Travel" bugs.
-  **Dataclass Contracts:** No more "Magic String" typos or blind dictionaries.
-  **Exception Flow:** The `run` method is now a clean, readable story instead of a maze of `if/else` checks.
-  **Performance Workers:** Math is separated from orchestration.
- Ret_1d, explicitly turns division-by-zero results (`inf`) into `NaN`, and replace [np.inf, -np.inf] with np.nan



v60  
- Converted code from notebook to modular system.
- Fixed divide by zero warning from calculate_gain
- Added subtitle to subplots
- Added Volatility Regime plot


v59  
- Removed "nest" of if-statements in **AlphaEngine.run**
- Use **Result Pattern** to handle errors
- Change verify_analyzer_short and verify_analyzer_long gain calculation from simple return to logarithmic return
- Change calculate_gain from simple return to logarithmic return
- Remove bfill from calculate_gain to prevent backfill with future data
- Verify macro_df calculation


v57, v58  
added marco subplotsThe macro regime framework is now fully documented with:
- Trend (SMA200 deviation) → Where we are in the cycle  
- Trend Velocity (Z) → How fast we're moving relative to normal
- VIX-Z → Market fear/complacency levels  

v56  

- De-coupled features_df and macro_df
- generate_features and audit_feature_engineering_integrity use GLOBAL_SETTINGS

v55  
Added
- audit_feature_engineering_integrity (check calculation in features_df)  

These are the metrics in plot  
- --- 1. LEGACY / SANITY CHECKS ---
- "Price Gain": lambda obs: QuantUtils.calculate_gain(obs["lookback_close"]),
- "Sharpe": lambda obs: QuantUtils.calculate_sharpe(obs["lookback_returns"]),
- "Sharpe (ATRP)": lambda obs: QuantUtils.calculate_sharpe_vol(
        obs["lookback_returns"], obs["atrp"]
    ),
- "Sharpe (TRP)": lambda obs: QuantUtils.calculate_sharpe_vol(
        obs["lookback_returns"], obs["trp"]
    ),
- --- 2. NEW QUANT METRICS ---
- "Momentum (21d)": lambda obs: obs["mom_21"],
- "Information Ratio (IR)": lambda obs: obs["ir_63"],  # Kept this one
- "Consistency (WinRate)": lambda obs: obs["consistency"],
- "Oversold (RSI)": lambda obs: -obs["rsi"],
- "Dip Buyer (Drawdown)": lambda obs: -obs["dd_21"],
- "Low Volatility": lambda obs: -obs["atrp"],







v54
-  **Replaced plot_walk_forward_analyzer with create_walk_forward_analyzer**


v53  
Looking at this registry with a quant lens, the list is **comprehensive but bloated**—we have **momentum measured five times under different names** (roc₁, roc₃, roc₅, roc₁₀, roc₂₁ and their negative twins “Pullback”).  
That’s **10 slots** telling us almost the same story at slightly different lags; in a rank-based engine they will **crowd the signal space** and inflate turnover without adding IC.

Duplicate / redundant cluster  
- Momentum 1 D ↔ Pullback 1 D (perfect mirror)  
- Same for 3 D, 5 D, 10 D, 21 D.  
**Keep one side only**—momentum is enough; the portfolio constructor can always **reverse the rank** if it wants “oversold”.

Close cousins that can be merged  
- “Sharpe” vs “Sharpe (ATRP)” – both are return / vol; keep **ATRP version** because it is regime-aware and smoother.  
- “RVol” vs “Vol_Regime” – both capture vol expansion; keep the **longer-memory one** (Vol_Regime) and drop the intraday snapshot.

Gaps that matter to a quant  
1. **Consistency sensor**: nowhere do we ask “how often did the ticker close higher than it opened?” – add **5-day win-rate** or **up-day hit-ratio**.  
2. **Risk-adjusted intraday strength**: no **Sharpe(on-balance volume)** or **volume-momentum efficiency**; OBV_Score is raw, not risk-scaled.  
3. **Benchmark-relative consistency**: “Alpha (RelStrength)” is cumulative; add **rolling information ratio vs SPY** to catch *sustained* alpha, not one gap.  
4. **Tail flag**: no **skew** or **max-drawdown** metric; a single 20 % gap stock can poison the book.  
5. **Macro regime overlay**: no **beta-to-SPY** or **correlation-break** sensor; mid-2022 macro swings showed that low-beta names behaved like a different asset class.

Recommended minimal clean set (≤ 12 metrics)

1. Sharpe(ATRP) – strategic anchor  
2. Momentum 21 D – slow trend  
3. Momentum 5 D – fast trend  
4. 5-day win-rate – consistency  
5. RSI(Trend) – strength confirmation  
6. OBV_Score – volume conviction  
7. Vol_Regime – vol expansion filter  
8. Alpha(RelStrength) 63-day IR – benchmark consistency  
9. Max 21-day drawdown – tail guard  
10. Beta-to-SPY – macro regime tag  

Drop everything else; the freed-up slots reduce collinearity, cut turnover, and leave head-room for **interaction terms** (e.g. momentum × consistency) that actually add orthogonal signal.



Below is a single, fully-vectorised block that adds the **five gap metrics** to your existing MultiIndex OHLCV frame.  
It never loops over tickers; everything is done with `groupby(level=0).rolling(...)` so it runs in C-speed and keeps the same index.

```python
import pandas as pd
import numpy as np

# ----------  CONFIG  -------------------------------------------------
LKB_RET   = 21          # look-back for return-based metrics
LKB_CONS  = 5           # consistency window (days)
LKB_IR    = 63          # IR window
LKB_BETA  = 63          # beta window
LKB_TAIL  = 21          # max-drawdown window
BENCH     = 'SPY'       # ticker that exists in your universe
# ---------------------------------------------------------------------

# 1.  DAILY RETURNS ----------------------------------------------------
df['ret'] = df.groupby(level=0)['Adj Close'].pct_change()

# 2.  CONSISTENCY SENSOR  (5-day win-rate) -----------------------------
df['up']  = df['ret'].gt(0).astype(int)
df['consistency_5d'] = (df.groupby(level=0)['up']
                          .rolling(LKB_CONS).mean()
                          .reset_index(level=0, drop=True))

# 3.  BENCHMARK-RELATIVE CONSISTENCY  (63-day IR vs SPY) ---------------
# need benchmark return
bench_ret = df.xs(BENCH, level=0)['ret'].rename('bench_ret')
df = df.join(bench_ret, how='left')          # broadcast to all tickers

df['active'] = df['ret'] - df['bench_ret']
g = df.groupby(level=0)
active_mean  = g['active'].rolling(LKB_IR).mean()
active_std   = g['active'].rolling(LKB_IR).std()
df['IR_63d'] = active_mean / active_std      # Information Ratio

# 4.  TAIL FLAG  (21-day max drawdown) ---------------------------------
roll_max = g['Adj Close'].rolling(LKB_TAIL).max()
dd = (df['Adj Close'] - roll_max) / roll_max
df['max_dd_21d'] = dd.groupby(level=0).rolling(LKB_TAIL).min()

# 5.  MACRO REGIME OVERLAY  (beta to SPY) ------------------------------
cov  = g['ret'].rolling(LKB_BETA).cov(df['bench_ret'])
var  = df['bench_ret'].groupby(level=0).rolling(LKB_BETA).var()
df['beta_SPY'] = cov / var

# 6.  RISK-ADJUSTED INTRADAY STRENGTH  (OBV Sharpe) --------------------
# OBV
df['close_chg'] = df.groupby(level=0)['Adj Close'].diff()
df['vol_dir']   = np.where(df['close_chg'] > 0,  df['Volume'],
                   np.where(df['close_chg'] < 0, -df['Volume'], 0))
df['obv'] = df.groupby(level=0)['vol_dir'].cumsum()

# OBV return & vol
df['obv_ret'] = df.groupby(level=0)['obv'].pct_change()
obv_mean = g['obv_ret'].rolling(LKB_RET).mean()
obv_std  = g['obv_ret'].rolling(LKB_RET).std()
df['OBV_Sharpe_21d'] = obv_mean / obv_std

# drop helper columns --------------------------------------------------
df.drop(columns=['up','bench_ret','active','close_chg','vol_dir'], inplace=True)
```

After the block you have five new columns:

- `consistency_5d`      – 5-day win-rate (0-1)  
- `IR_63d`              – 63-day Information Ratio vs SPY  
- `max_dd_21d`          – 21-day maximum drawdown (≤ 0)  
- `beta_SPY`            – rolling beta to SPY  
- `OBV_Sharpe_21d`      – OBV risk-adjusted momentum  

All are aligned to the original MultiIndex and ready to be ranked or z-scored inside your Alpha Engine.

v52  
- **Cascase Filter results `AGREED` with bot_v54i.ipynb**
- **Cascade Filter works with df_ohlcv_subset**
- **verify_engine_results_short_form**
- **verify_engine_results_long_form**
-  **The Temporal Alignment Fix:** We synchronized the "Reward" (Returns) and "Risk" (Volatility) by implementing the $N-1$ denominator logic. This ensures that Day 1's volatility no longer dilutes your Sharpe scores.
-  **The Event-Driven Re-normalization:** We verified that the Engine correctly resets capital and weights at the start of the Holding period, giving you an accurate "Fresh Start" performance metric.
-  **The Double-Blind Verification:** We proved that the Engine's True Range (TRP) math is flawless by recreating it from raw High/Low/Close data and achieving an 8-decimal match.
-  **Mathematical Fortification:** We centralized all logic into a polymorphic `QuantUtils` kernel that handles both single-portfolio reports and whole-universe rankings with built-in numerical safety.
-  **Volatility Evolution:** We successfully added `TRP` (True Range Percent) and the `Sharpe (TRP)` metric, giving you a raw, high-frequency alternative to the smoothed ATR.
-  **Data Integrity:** We implemented the "Momentum Collapse" tripwire (`verify_ranking_integrity`) to ensure that your risk-adjusted rankings never accidentally devolve into simple price momentum.
-  **The "Audit Pack" Architecture:** We collapsed fragmented results into a single, atomic container, ensuring that your inputs, results, and debug data are always perfectly synchronized.
-  **Total Transparency:** We replaced scattered CSV files with a unified **Excel Audit Report**, allowing for 1-to-1 manual verification of every calculation in the system.



v51

UNDO v50, Calculate Sharpe(ATR) using mean over lookback period.  

Comment out ``# --- PINPOINT START: ATRP SWITCH ---`` in function ``_select_tickers`` can switch between ``Averaged ATRP over lookback period`` and ``Current ATRP``  
    # --- PINPOINT START: ATRP SWITCH ---  
    # To switch between Old (Averaged ATRP) and New (Current ATRP):  
    # 1. Comment out the logic you DON'T want.  
    # 2. Uncomment the logic you DO want.  


v50

Ticker selection based on atrp_value_for_obs based on decision day, was based on average over lookback period. 

v48  
### Summary of what you just accomplished:
1.  **Strict Math:** `QuantUtils` now contains an `assert` that prevents any dev (or AI) from filling the first day with 0.0.
2.  **Semantic Protection:** Variables are now named `returns_WITH_BOUNDARY_NAN`, signaling to the AI that the Null value is part of its identity.
3.  **Complete SOLID Separation:** The Engine CONDUCTS the simulation, while `QuantUtils` CALCULATES the results. They no longer share logic.

**1. Data Flow of `plot_walk_forward_analyzer`**
The function acts as a **UI wrapper** around the `AlphaEngine` class. The flow is:
1.  **Input:** User selects parameters (Dates, Lookback, Strategy).
2.  **State Construction:** `AlphaEngine` slices the historical data (`df_ohlcv`, `df_atrp`) up to the `decision_date`.
3.  **Policy Execution (Hardcoded):** The engine applies the logic (e.g., `METRIC_REGISTRY['Sharpe']`) to rank stocks based *only* on the Lookback window.
4.  **Environment Step:** It simulates a "Buy" at `decision_date + 1` and calculates the returns over the `holding_period`.
5.  **Reward Generation:** It outputs performance metrics (`holding_p_gain`, `holding_p_sharpe`).

In [33]:
# 1. Enable Autoreload
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

def add_project_root_to_path():
    """Find notebooks_RLVR and add to sys.path."""
    current = Path.cwd()

    # Search upward for notebooks_RLVR folder
    for path in [current] + list(current.parents):
        if path.name == "notebooks_RLVR":
            sys.path.insert(0, str(path))
            print(f"✓ Added to path: {path}")
            return path
        # Also check if notebooks_RLVR exists as child (for running from stocks/)
        candidate = path / "notebooks_RLVR"
        if candidate.exists():
            sys.path.insert(0, str(candidate))
            print(f"✓ Added to path: {candidate}")
            return candidate

    raise RuntimeError("Could not find notebooks_RLVR directory")


# Run once at notebook start
add_project_root_to_path()


# 2. Force reload cached modules (run this to refresh code changes)
# import importlib

modules_to_reload = [
    "core.analyzer",
    "core.auditor",
    "core.contracts",    
    "core.engine",
    "core.features",
    "core.paths",
    "core.performance",
    "core.quant",
    "core.result",
    "core.settings",
    "core.utils",
    "strategy.registry",
]

for mod in modules_to_reload:
    if mod in sys.modules:
        del sys.modules[mod]


# 3. Standard imports
import pandas as pd
# import os
import numpy as np

from IPython.display import display
# from dataclasses import fields, asdict, is_dataclass
# from typing import List, Union, Tuple 


# 4. Fresh imports (these will re-import from disk due to cache clearing above)

from core.analyzer import create_walk_forward_analyzer, run_headless_simulation
from core.auditor import SystemAuditor as SA
from core.contracts import FilterPack
from core.engine import AlphaEngine
from core.features import generate_features
from core.paths import OUTPUT_DIR
# from core.performance import PerformanceCalculator
# from core.quant import QuantUtils
from core.result import HeadlessReporter
from core.settings import GLOBAL_SETTINGS
from core.utils import SystemUtils as SU
# from strategy.registry import METRIC_REGISTRY


# 5. Pandas display settings
pd.set_option("display.max_rows", 100)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 1000)
pd.set_option("display.max_colwidth", 50)
pd.set_option("display.precision", 4)


# # 6. Instantiate engine (customize DataFrames as needed)
# old_master_engine = AlphaEngine(
#     df_ohlcv=df_ohlcv,
#     features_df=features_df,
#     macro_df=macro_df,
#     df_close_wide=df_close_wide,
#     df_atrp_wide=df_atrp_wide,
#     df_trp_wide=df_trp_wide,
# )

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
✓ Added to path: c:\Users\ping\Files_win10\python\py311\stocks\notebooks_RLVR
NOTEBOOKS_RLVR_ROOT: C:\Users\ping\Files_win10\python\py311\stocks\notebooks_RLVR

OUTPUT_DIR: C:\Users\ping\Files_win10\python\py311\stocks\notebooks_RLVR\output



In [4]:
from dotenv import load_dotenv
import os
from pathlib import Path


def load_env_from_root(env_var_name):
    """
    Load specified environment variable from .env file in root directory

    Parameters:
    -----------
    env_var_name : str
        Name of the environment variable to retrieve (e.g., 'DATA_PATH_OHLCV', 'API_KEY')

    Returns:
    --------
    str : Value of the requested environment variable

    Example:
    --------
    DATA_PATH_OHLCV = load_env_from_root('DATA_PATH_OHLCV')
    API_KEY = load_env_from_root('API_KEY')
    """
    # Start from current file or current working directory
    try:
        start_path = Path(__file__).resolve().parent
    except NameError:
        # If __file__ not defined (e.g., Jupyter), use current working directory
        start_path = Path.cwd()

    # Search upwards for the .env directory
    current = start_path
    for _ in range(10):  # Limit search depth
        env_path = current / ".env" / "my_api_key.env"
        if env_path.exists():
            load_dotenv(env_path, override=True)
            value = os.getenv(env_var_name)
            if value is None:
                raise KeyError(f"Variable '{env_var_name}' not found in {env_path}")
            return value

        # Go up one level
        parent = current.parent
        if parent == current:  # Reached root
            break
        current = parent

    raise FileNotFoundError(
        f"Could not find .env/my_api_key.env when searching for '{env_var_name}'"
    )


#

In [5]:
DATA_PATH_OHLCV = load_env_from_root("DATA_PATH_OHLCV")
DATA_PATH_INDICES = load_env_from_root("DATA_PATH_INDICES")
print(f"DATA_PATH_OHLCV: {DATA_PATH_OHLCV}")
print(f"DATA_PATH_INDICES: {DATA_PATH_INDICES}")

DATA_PATH_OHLCV: c:\Users\ping\Files_win10\python\py311\stocks\data\df_OHLCV_stocks_etfs.parquet
DATA_PATH_INDICES: c:\Users\ping\Files_win10\python\py311\stocks\data\df_indices.parquet


In [6]:
df_indices = pd.read_parquet(DATA_PATH_INDICES, engine="pyarrow")
print(f"df_indices:|n{df_indices}")

df_indices:|n                   Adj Open  Adj High  Adj Low  Adj Close  Volume
Ticker Date                                                      
^BSESN 1997-07-01   4263.11   4301.77  4247.66    4300.86       0
       1997-07-02   4302.96   4395.31  4295.40    4333.90       0
       1997-07-03   4335.79   4393.29  4299.97    4323.46       0
       1997-07-04   4332.70   4347.59  4300.58    4323.82       0
       1997-07-07   4326.81   4391.01  4289.49    4291.45       0
...                     ...       ...      ...        ...     ...
^VIX3M 2026-03-16     25.98     25.98    24.67      24.92       0
       2026-03-17     24.26     24.58    23.87      24.33       0
       2026-03-18     25.00     26.57    24.82      26.56       0
       2026-03-19     27.96     28.03    25.19      25.54       0
       2026-03-20     26.07     28.61    25.91      27.43       0

[136293 rows x 5 columns]


In [ ]:
_indices = df_indices.index.get_level_values(0).unique().tolist()
display(_indices)
df_indices.info()

In [7]:
print(f"Takes about 1.5 minutes")
df_ohlcv = pd.read_parquet(DATA_PATH_OHLCV, engine="pyarrow")

Takes about 1.5 minutes


In [ ]:
print(f"df_ohlcv.head():\n {df_ohlcv.head()}\n")
df_ohlcv.info()

In [8]:
print(f"Takes about 2.5 minutes to generate_features")

features_df, macro_df = generate_features(
    df_ohlcv=df_ohlcv,
    df_indices=df_indices,
    benchmark_ticker=GLOBAL_SETTINGS["benchmark_ticker"],
)

Takes about 2.5 minutes to generate_features
⚡ Generating Decoupled Features (Benchmark: SPY)...


In [ ]:
print(f"features_df.info():\n{features_df.info()}\n")
print(f"features_df.index.names:\n{features_df.index.names}\n")
print(f"macro_df.info():\n{macro_df.info()}\n")
print(f"macro_df.index.names:\n{macro_df.index.names}\n")

In [ ]:
# Pre-flight Automated Audit Suite
checks = [
    SA.verify_math_integrity(),
    SA.verify_ranking_integrity(),
    SA.verify_vol_alignment_integrity(),
    SA.verify_feature_engineering_integrity(),
]

for check in checks:
    icon = "✅" if check.ok else "🔥"
    print(f"{icon} {check.msg}")
    if not check.ok:
        print("🛑 Critical Failure. System Halted.")
        break

print("=" * 85)
# Separate verify_marco_engine output from intertwine with other outputs

checks = [
    SA.verify_macro_engine(
        df_ohlcv=df_ohlcv,
        df_indices=df_indices,
        original_macro_df=macro_df,
        settings=GLOBAL_SETTINGS,
    ),
]

for check in checks:
    icon = "✅" if check.ok else "🔥"
    print(f"{icon} {check.msg}")
    if not check.ok:
        print("🛑 Critical Failure. System Halted.")
        break

In [9]:
print(
    "🚀 Generating Wide Matrices for Instant Backtesting... (takes about 1 minute to run)"
)

# 1. Price Matrix
df_close_wide = df_ohlcv["Adj Close"].unstack(level=0)

# 2. Volatility Matrices (Unstack and Align)
# Using features_df (the first item from the tuple)
print("   - Unstacking ATRP...")
df_atrp_wide = features_df["ATRP"].unstack(level=0).reindex_like(df_close_wide)

print("   - Unstacking TRP...")
df_trp_wide = features_df["TRP"].unstack(level=0).reindex_like(df_close_wide)

# 3. Handle Data Gaps (Sanitize the Wide Matrices)
if GLOBAL_SETTINGS["handle_zeros_as_nan"]:
    df_close_wide = df_close_wide.replace(0, np.nan)

# Forward fill up to the limit, then fill remaining with the "Disaster Detection" value
df_close_wide = df_close_wide.ffill(limit=GLOBAL_SETTINGS["max_data_gap_ffill"])
df_close_wide = df_close_wide.fillna(GLOBAL_SETTINGS["nan_price_replacement"])

print(
    f"✅ Pre-computation Complete. Tickers: {len(df_close_wide.columns)}, Days: {len(df_close_wide)}"
)
print("   Ready: df_close_wide, df_atrp_wide, df_trp_wide, and macro_df.")

🚀 Generating Wide Matrices for Instant Backtesting... (takes about 1 minute to run)
   - Unstacking ATRP...
   - Unstacking TRP...
✅ Pre-computation Complete. Tickers: 1582, Days: 16162
   Ready: df_close_wide, df_atrp_wide, df_trp_wide, and macro_df.


In [10]:
# 6. Instantiate engine (customize DataFrames as needed)
old_master_engine = AlphaEngine(
    df_ohlcv=df_ohlcv,
    features_df=features_df,
    macro_df=macro_df,
    df_close_wide=df_close_wide,
    df_atrp_wide=df_atrp_wide,
    df_trp_wide=df_trp_wide,
)

In [ ]:
# universe_subset=None means "Scan the whole market"
analyzer1, stage1_pack = create_walk_forward_analyzer(
    old_master_engine, universe_subset=None
)

print("🚀 Ready for Stage 1: Run Simulation for first filter.")
analyzer1.show()

🚀 Ready for Stage 1: Run Simulation for first filter.
DEBUG: 937 stocks passed filters on 2025-12-10


In [ ]:
# Get decision date from last run
decision_date_last_run = FilterPack(decision_date=analyzer1.last_run.decision_date)

# 1. LAUNCH STAGE 2 (Cascade)
# universe_subset=analyzer1.last_run.tickers means "Scan the whole market"
analyzer2, stage1_pack = create_walk_forward_analyzer(
    old_master_engine,
    universe_subset=analyzer1.last_run.tickers,
    # universe_subset=None,
    filter_pack=decision_date_last_run,
)

print("🚀 Ready for Stage 2: Run Simulation for 2nd filter.")
analyzer2.show()

In [ ]:
###############################
###############################

In [12]:
my_analyzer = analyzer1

my_res = SU.visualize_analyzer_structure(my_analyzer)

🔍 HIGH-TRANSPARENCY AUDIT MAP
[  0] 📦 audit_pack (EngineOutput)
[  1]   📈 portfolio_series (shape=(28,))
[  2]   📈 benchmark_series (shape=(28,))
[  3]   🧮 normalized_plot_data (shape=(28, 2))
[  4]   📂 tickers (len=2)
[  5]     📄 index_0 (str)
[  6]     📄 index_1 (str)
[  7]   📈 initial_weights (shape=(2,))
[  8]   📂 perf_metrics (len=24)
[  9]     🔢 full_p_gain (float)
[ 10]     🔢 full_p_sharpe (float)
[ 11]     🔢 full_p_sharpe_atrp (float)
[ 12]     🔢 full_p_sharpe_trp (float)
[ 13]     🔢 lookback_p_gain (float)
[ 14]     🔢 lookback_p_sharpe (float)
[ 15]     🔢 lookback_p_sharpe_atrp (float)
[ 16]     🔢 lookback_p_sharpe_trp (float)
[ 17]     🔢 holding_p_gain (float)
[ 18]     🔢 holding_p_sharpe (float)
[ 19]     🔢 holding_p_sharpe_atrp (float)
[ 20]     🔢 holding_p_sharpe_trp (float)
[ 21]     🔢 full_b_gain (float)
[ 22]     🔢 full_b_sharpe (float)
[ 23]     🔢 full_b_sharpe_atrp (float)
[ 24]     🔢 full_b_sharpe_trp (float)
[ 25]     🔢 lookback_b_gain (float)
[ 26]     🔢 lookback_b

In [13]:
SU.peek(3, my_res)

 📍 INDEX: [3]
 🏷️  NAME:  normalized_plot_data
 📂 PATH:  audit_pack -> normalized_plot_data



Ticker,ONON,TD
Date,,
2025-11-10,1.0000,1.0000
2025-11-11,1.0063,1.0057
2025-11-12,1.1874,1.0197
2025-11-13,1.2128,1.0026
2025-11-14,1.2128,1.0026
2025-11-17,1.1842,1.0038
2025-11-18,1.2203,1.0138
2025-11-19,1.2386,1.0055
2025-11-20,1.1848,1.0022


Ticker,ONON,TD
Date,,
2025-11-10,1.0000,1.0000
2025-11-11,1.0063,1.0057
2025-11-12,1.1874,1.0197
2025-11-13,1.2128,1.0026
2025-11-14,1.2128,1.0026
2025-11-17,1.1842,1.0038
2025-11-18,1.2203,1.0138
2025-11-19,1.2386,1.0055
2025-11-20,1.1848,1.0022


In [14]:
analyzer1.last_run.tickers
analyzer1.last_run.start_date
analyzer1.last_run.holding_end_date

Timestamp('2025-12-18 00:00:00')

In [15]:
# 3. Post-flight Reconciliation
audit = SA.verify_analyzer_short(my_analyzer)
if not audit.ok:
    print(f"🚨 ALERT: {audit.msg}")
    # You could trigger a notification or log the failure here


***********************************************************************************************
🕵️  STARTING SHORT-FORM AUDIT: Sharpe (ATRP) @ 2025-12-10
⚠️  ASSUMPTION: Verification logic is independent, but trusts Engine source DataFrames
    (engine.features_df, engine.df_close, and debug['portfolio_raw_components'])
***********************************************************************************************
🕵️  AUDIT: Sharpe (ATRP) @ 2025-12-10
LAYER 1: SURVIVAL  | Universe: 1580 -> Survivors: 937 | ✅ PASS
LAYER 2: SELECTION | Strategy: Sharpe (ATRP) | Selection Match: ✅ PASS
LAYER 3: PERFORMANCE (Holding Period: 5 days)
Metric               | Engine       | Manual       | Status
-----------------------------------------------------------------------------------------------
Gain                 |    -0.006771 |    -0.006771 | ✅ PASS
Sharpe               |    -2.486459 |    -2.486459 | ✅ PASS
Sharpe (ATRP)        |    -0.051518 |    -0.051518 | ✅ PASS
Sharpe (TRP)         |    -

In [16]:
audit = SA.verify_analyzer_long(my_analyzer, n_tickers=5)


🛡️  STARTING NUCLEAR AUDIT | 2025-12-10 | Sharpe (ATRP)
*************************************************************************************
⚠️  ASSUMPTION: Verification logic is independent, but trusts Engine source DataFrames
    (engine.features_df, engine.df_close, and debug['portfolio_raw_components'])
*************************************************************************************
📝 1. PERFORMANCE RECONCILIATION


Period                     Full Holding Lookback
Entity    Metric                                
Benchmark Gain           ✅ PASS  ✅ PASS   ✅ PASS
          Sharpe         ✅ PASS  ✅ PASS   ✅ PASS
          Sharpe (ATRP)  ✅ PASS  ✅ PASS   ✅ PASS
          Sharpe (TRP)   ✅ PASS  ✅ PASS   ✅ PASS
Group     Gain           ✅ PASS  ✅ PASS   ✅ PASS
          Sharpe         ✅ PASS  ✅ PASS   ✅ PASS
          Sharpe (ATRP)  ✅ PASS  ✅ PASS   ✅ PASS
          Sharpe (TRP)   ✅ PASS  ✅ PASS   ✅ PASS


📝 2. SURVIVAL AUDIT
   Survival Integrity: ✅ PASS (Engine: 937 vs Auditor: 937)

📝 3. UNIVERSAL SELECTION AUDIT | Strategy: Sharpe (ATRP)
   Scope: Evaluated 937 candidates (Full Universe).
   Result: 937 PASSED | 0 FAILED
   All scores match registry math. Sharpe (ATRP) results of the first 5 tickers


,Ticker,Engine,Manual,Delta,Status
Rank,,,,,
1,A,-0.02991409,-0.02991409,0.00000000,✅ PASS
2,AA,0.15969722,0.15969722,0.00000000,✅ PASS
3,AAL,0.15552532,0.15552532,0.00000000,✅ PASS
4,AAPL,0.08501287,0.08501287,0.00000000,✅ PASS
5,ABBV,0.06503462,0.06503462,0.00000000,✅ PASS


In [17]:
# Takes 4 seconds to run, checks selected tickers from analyzer1
SA.audit_feature_engineering_integrity(my_analyzer, mode="last_run")


🕵️  NUCLEAR FEATURE AUDIT | Mode: LAST_RUN | Tickers: 2
STEP 1: BOUNDARY INTEGRITY   | MultiIndex Isolation Check | ✅ PASS
STEP 2: SHADOW CALCULATIONS  | Re-computing metrics... DONE (0.18s)

Metric               | Max Delta    | Correlation  | Status
-------------------------------------------------------------------------------------
Ret_1d               |   0.0000e+00 |     1.000000 | ✅ PASS
ATR                  |   0.0000e+00 |     1.000000 | ✅ PASS
ATRP                 |   0.0000e+00 |     1.000000 | ✅ PASS
TRP                  |   0.0000e+00 |     1.000000 | ✅ PASS
RSI                  |   0.0000e+00 |     1.000000 | ✅ PASS
Mom_21               |   0.0000e+00 |     1.000000 | ✅ PASS
Consistency          |   0.0000e+00 |     1.000000 | ✅ PASS
DD_21                |   0.0000e+00 |     1.000000 | ✅ PASS
RollingStalePct      |   0.0000e+00 |     1.000000 | ✅ PASS
RollMedDollarVol     |   0.0000e+00 |     1.000000 | ✅ PASS
RollingSameVolCount  |   0.0000e+00 |     1.000000 | ✅ PASS
M

### Audit Every Tickers in features_df, Takes 3 minutes 

In [18]:
# # Takes 3 minutes to run, checks every tickers ~1580 tickers
# SA.audit_feature_engineering_integrity(my_analyzer, mode="system")

### Export Ticker's OHLCV and Features

In [19]:
f_name_excel = OUTPUT_DIR / "Audit_Verification_Report.xlsx"

SU.export_audit_to_excel(audit_pack=my_analyzer.last_run, filename=f_name_excel)

📂 [EXCEL AUDIT] Building full transparency report: C:\Users\ping\Files_win10\python\py311\stocks\notebooks_RLVR\output\Audit_Verification_Report.xlsx
✨ Audit Report Complete: C:\Users\ping\Files_win10\python\py311\stocks\notebooks_RLVR\output\Audit_Verification_Report.xlsx


WindowsPath('C:/Users/ping/Files_win10/python/py311/stocks/notebooks_RLVR/output/Audit_Verification_Report.xlsx')

In [20]:
f_name_csv = OUTPUT_DIR / "all_tickers_data_stacked.csv"

# Single call replaces your 3 cells
file_path = SU.export_last_run_tickers_data_to_csv(
    analyzer=my_analyzer,
    df_ohlcv=df_ohlcv,
    features_df=features_df,
    filename=f_name_csv,
)

Creating combined dictionary for 3 ticker(s)
Date range: 2025-11-10 00:00:00 to 2025-12-18 00:00:00
Data retrieved for 3 ticker(s) from 2025-11-10 00:00:00 to 2025-12-18 00:00:00
Total rows: 84
Date range in data: 2025-11-10 00:00:00 to 2025-12-18 00:00:00
  TD: 28 rows
  ONON: 28 rows
  SPY: 28 rows
Features data retrieved for 3 ticker(s) from 2025-11-10 00:00:00 to 2025-12-18 00:00:00
Total rows: 84
Date range in data: 2025-11-10 00:00:00 to 2025-12-18 00:00:00
Available features: ATR, ATRP, TRP, RSI, Mom_21, Consistency, IR_63, Beta_63, DD_21, Ret_1d, RollingStalePct, RollMedDollarVol, RollingSameVolCount
  TD: 28 rows
  ONON: 28 rows
  SPY: 28 rows

Processing TD...
  ✓ Successfully combined data
  OHLCV shape: (28, 5)
  Features shape: (28, 13)
  Combined shape: (28, 18)
  Date range: 2025-11-10 00:00:00 to 2025-12-18 00:00:00

Processing ONON...
  ✓ Successfully combined data
  OHLCV shape: (28, 5)
  Features shape: (28, 13)
  Combined shape: (28, 18)
  Date range: 2025-11-10 00:

In [21]:
SU.create_combined_dict(
    df_ohlcv=df_ohlcv,
    features_df=features_df,
    tickers=my_analyzer.last_run.tickers,
    date_start=my_analyzer.last_run.start_date,
    date_end=analyzer1.last_run.holding_end_date,
    verbose=False,
)

{'TD':             Adj Open  Adj High  Adj Low  Adj Close   Volume     ATR    ATRP     TRP      RSI  Mom_21  Consistency   IR_63  Beta_63   DD_21  Ret_1d  RollingStalePct  RollMedDollarVol  RollingSameVolCount
 Date                                                                                                                                                                                                      
 2025-11-10     81.17     81.28    80.51      81.14  1537700  1.1176  0.0138  0.0095  55.1221  0.0360          0.6  0.0657   0.5685 -0.0191  0.0030              0.0        1.1286e+08                  0.0
 2025-11-11     81.25     81.99    81.25      81.60  2429300  1.0985  0.0135  0.0104  57.9484  0.0346          0.8  0.0505   0.5582 -0.0135  0.0057              0.0        1.1286e+08                  0.0
 2025-11-12     81.80     82.93    81.80      82.74  1724400  1.1151  0.0135  0.0161  63.9994  0.0422          0.8  0.0694   0.5562  0.0000  0.0140              0.0        1.1297

In [22]:
_tic = "NVDA"
_start_date = "2025-03-12"
_end_date = "2026-03-12"
# print(features_df.loc[_tic][_start_date:_end_date])
print(features_df.loc[_tic].tail(10))
# features_df.loc[_tic][_start_date:_end_date][["Ret_1d", "Consistency"]]

               ATR    ATRP     TRP      RSI  Mom_21  Consistency   IR_63  Beta_63   DD_21  Ret_1d  RollingStalePct  RollMedDollarVol  RollingSameVolCount
Date                                                                                                                                                     
2026-03-09  6.4513  0.0353  0.0402  47.8271  0.0627          0.6  0.0139   2.1743 -0.0660  0.0272              0.0        3.0610e+10                  0.0
2026-03-10  6.3069  0.0341  0.0240  50.0459 -0.0035          0.8  0.0316   2.1737 -0.0552  0.0116              0.0        3.0610e+10                  0.0
2026-03-11  6.0828  0.0327  0.0170  51.3799 -0.0210          0.8  0.0212   2.1921 -0.0487  0.0069              0.0        3.0548e+10                  0.0
2026-03-12  5.9541  0.0325  0.0234  48.2240 -0.0286          0.6  0.0229   2.1141 -0.0635 -0.0155              0.0        3.0283e+10                  0.0
2026-03-13  5.9681  0.0331  0.0341  45.2320 -0.0515          0.6  0.0255   2

In [23]:
# analyzer1.last_run.tickers
# analyzer1.last_run.start_date
# analyzer1.last_run.holding_end_date
_tic = analyzer1.last_run.tickers[0]
_start_date = analyzer1.last_run.start_date
_end_date = analyzer1.last_run.holding_end_date

In [24]:
result = SU.create_combined_dict(
    df_ohlcv=df_ohlcv,
    features_df=features_df,
    tickers=[_tic],
    date_start=_start_date,
    date_end=_end_date,
    verbose=False,
)
print(f"ticker: {_tic}\n")
print(result[_tic])

ticker: TD

            Adj Open  Adj High  Adj Low  Adj Close   Volume     ATR    ATRP     TRP      RSI  Mom_21  Consistency   IR_63  Beta_63   DD_21  Ret_1d  RollingStalePct  RollMedDollarVol  RollingSameVolCount
Date                                                                                                                                                                                                      
2025-11-10     81.17     81.28    80.51      81.14  1537700  1.1176  0.0138  0.0095  55.1221  0.0360          0.6  0.0657   0.5685 -0.0191  0.0030              0.0        1.1286e+08                  0.0
2025-11-11     81.25     81.99    81.25      81.60  2429300  1.0985  0.0135  0.0104  57.9484  0.0346          0.8  0.0505   0.5582 -0.0135  0.0057              0.0        1.1286e+08                  0.0
2025-11-12     81.80     82.93    81.80      82.74  1724400  1.1151  0.0135  0.0161  63.9994  0.0422          0.8  0.0694   0.5562  0.0000  0.0140              0.0        1.129

In [ ]:
#######################
#######################

In [ ]:
# import pandas as pd
# from typing import Dict, Any
# from core.contracts import EngineOutput, EngineInput


# class HeadlessReporter:
#     """
#     Expert Logic: Extracts metrics and timeline metadata from EngineOutput.
#     Maintains transparency and trust without needing Plotly.
#     """

#     @staticmethod
#     def get_metadata(res: EngineOutput) -> Dict[str, Any]:
#         """Extracts the timeline and ticker list as structured data."""
#         return {
#             "start": res.start_date.date(),
#             "decision": res.decision_date.date(),
#             "entry": res.buy_date.date(),
#             "end": res.holding_end_date.date(),
#             "tickers": res.tickers,
#             "ticker_count": len(res.tickers),
#         }

#     @staticmethod
#     def get_metrics_table(res: EngineOutput) -> pd.DataFrame:
#         """The existing table logic—keeps the math verified."""
#         m = res.perf_metrics
#         rows = []
#         metric_types = [
#             ("Gain", "gain"),
#             ("Sharpe", "sharpe"),
#             ("Sharpe (ATRP)", "sharpe_atrp"),
#             ("Sharpe (TRP)", "sharpe_trp"),
#         ]

#         for label, key in metric_types:
#             p_row = {
#                 "Metric": f"Group {label}",
#                 "Full": m.get(f"full_p_{key}"),
#                 "Lookback": m.get(f"lookback_p_{key}"),
#                 "Holding": m.get(f"holding_p_{key}"),
#             }
#             b_row = {
#                 "Metric": f"Benchmark {label}",
#                 "Full": m.get(f"full_b_{key}"),
#                 "Lookback": m.get(f"lookback_b_{key}"),
#                 "Holding": m.get(f"holding_b_{key}"),
#             }

#             d_row = {"Metric": f"== {label} Delta"}
#             for col in ["Full", "Lookback", "Holding"]:
#                 p_val, b_val = p_row[col] or 0.0, b_row[col] or 0.0
#                 d_row[col] = p_val - b_val
#             rows.extend([p_row, b_row, d_row])

#         return pd.DataFrame(rows).set_index("Metric")


# def run_headless_simulation(engine, inputs: EngineInput) -> pd.DataFrame:
#     """
#     Orchestrator: Prints the Audit Timeline and Returns the Metrics DataFrame.
#     """
#     result = engine.run(inputs)
#     if result.error_msg:
#         print(f"🚨 Engine Error: {result.error_msg}")
#         return pd.DataFrame()

#     # 1. Extract and Print Metadata (Matches the Screenshot Audit)
#     meta = HeadlessReporter.get_metadata(result)

#     print("-" * 70)
#     print(
#         f"Timeline: [{meta['start']}] -> Decision: {meta['decision']} -> "
#         f"Entry: {meta['entry']} -> End: {meta['end']}"
#     )

#     ticker_str = ", ".join(meta["tickers"])
#     print(f"Selected Tickers ({meta['ticker_count']}):\n{ticker_str}")
#     print("-" * 70)

#     # 2. Return Table
#     return HeadlessReporter.get_metrics_table(result)


#

In [ ]:
# 1. Define the Action (Inputs)
my_input = EngineInput(
    mode="Ranking",
    start_date=pd.Timestamp("2025-12-10"),  # Decision Date from your screenshot
    lookback_period=21,
    holding_period=5,
    metric="Sharpe (ATRP)",
    benchmark_ticker="SPY",
    rank_start=11,
    rank_end=12,  # Choosing top 2 tickers as seen in your image (TD, ONON)
    debug=True,
)

# 2. Run the Headless Engine
metrics_df = run_headless_simulation(old_master_engine, my_input)

# 3. Verify Result
print("--- HEADLESS METRICS REPORT ---")
display(metrics_df)

# TEST: Gain in Holding Period (The Reward for RL)
reward = metrics_df.loc["Group Gain", "Holding"]
print(f"\nTarget Reward Signal: {reward:.4f}")

DEBUG: 937 stocks passed filters on 2025-12-10
----------------------------------------------------------------------
Timeline: [2025-11-10] -> Decision: 2025-12-10 -> Entry: 2025-12-11 -> End: 2025-12-18
Selected Tickers (2):
TD, ONON
----------------------------------------------------------------------
--- HEADLESS METRICS REPORT ---


,Full,Lookback,Holding
Metric,,,
Group Gain,0.2337,0.2304,-0.0068
Benchmark Gain,-0.0073,0.0090,-0.0186
== Gain Delta,0.2410,0.2215,0.0118
Group Sharpe,6.4330,7.4510,-2.4865
Benchmark Sharpe,-0.4945,0.9141,-7.6352
== Sharpe Delta,6.9275,6.5369,5.1487
Group Sharpe (ATRP),0.2936,0.3612,-0.0515
Benchmark Sharpe (ATRP),-0.0200,0.0367,-0.3363
== Sharpe (ATRP) Delta,0.3136,0.3245,0.2847



Target Reward Signal: -0.0068
